# Huấn Luyện AI Giải Captcha (solve-captcha) trên Google Colab

Jupyter Notebook này hướng dẫn bạn cách huấn luyện mô hình PyTorch giải captcha 4 ký tự sử dụng **GPU miễn phí (T4)** trên Google Colab để tăng tốc độ huấn luyện gấp hàng chục lần so với CPU.

### 📋 Quy trình chuẩn bị trước khi chạy:
1. Từ C# SaaS, gọi API `/api/v1/dataset/generate?count=4000` để sinh 4.000 ảnh captcha.
2. Tải file ZIP dataset tại `/api/v1/dataset/export` về máy tính (hoặc lấy file `captcha_dataset.zip` trong thư mục scratch của Agent).
3. Nén thư mục mã nguồn `solve-captcha` thành file `solve-captcha.zip`.
4. Upload cả hai file `solve-captcha.zip` và `captcha_dataset.zip` lên thư mục gốc của Google Colab.

## 🛠️ Bước 1: Kiểm tra kết nối GPU
Nhấp vào thanh thực đơn **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** làm Hardware accelerator trước khi chạy ô dưới.

In [ ]:
import torch
print("GPU khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tên GPU:", torch.cuda.get_device_name(0))

## 📦 Bước 2: Giải nén mã nguồn & Cài đặt thư viện

In [ ]:
# 1. Giải nén mã nguồn
!unzip -o -q solve-captcha.zip -d ./

# 2. Di chuyển các file từ thư mục con ra ngoài nếu bị bọc
# %cd solve-captcha (Bỏ dấu # ở đầu dòng này nếu mã nguồn của bạn nằm hoàn toàn trong thư mục solve-captcha)

# 3. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt

## 📊 Bước 3: Giải nén & Phân chia Dataset thành Train/Test
Script `split_dataset.py` sẽ tự động phân tách 4.000 ảnh trong file ZIP thành **3.000 ảnh Train** (lưu ở `data/train`) và **1.000 ảnh Test** (lưu ở `data/test`), kèm theo các file CSV nhãn tương ứng.

In [ ]:
# Chạy phân chia tập dữ liệu
!python split_dataset.py --zip captcha_dataset.zip

## 🚀 Bước 4: Chạy huấn luyện mô hình AI (PyTorch)
Chúng ta sẽ chạy huấn luyện mô hình CNN ResNet18 Multi-Head với **15 Epochs**.

In [ ]:
# Chạy huấn luyện bằng script train.py
!python src/train.py --epochs 15 --batch_size 64 --lr 0.001 --data_dir data --model_dir model

## 🔮 Bước 5: Chạy thử nghiệm nhận diện với một ảnh ngẫu nhiên
Kiểm tra xem mô hình AI giải thử một ảnh trong tập Test có chính xác không.

In [ ]:
import os
import glob

# Lấy file ảnh đầu tiên trong tập test để test thử
test_images = glob.glob("data/test/*.png")
if test_images:
    sample_image = test_images[0]
    print(f"🔍 Đang test thử với ảnh: {sample_image}")
    !python src/predict.py --image "{sample_image}" --model_path model/captcha_model.pth --alphabet_path model/alphabet.json
else:
    print("❌ Không tìm thấy ảnh test nào để chạy dự đoán!")

## 💾 Bước 6: Tải mô hình đã train về máy tính
Tải file trọng số mô hình `captcha_model.pth` và bảng ký tự `alphabet.json` về máy tính của bạn để sử dụng tích hợp vào hệ thống C# SaaS hoặc chạy demo.

In [ ]:
from google.colab import files
import os

if os.path.exists("model/captcha_model.pth"):
    files.download("model/captcha_model.pth")
    files.download("model/alphabet.json")
    print("🟢 Đang yêu cầu tải xuống files mô hình...")
else:
    print("❌ Không tìm thấy file model.pth để tải về!")